# 06 - Modelos

Acá entrenamos modelos que **aprenden** con los datos de `train`, y los comparamos contra la regla a superar del notebook 05: `mediana_4sem`, que en validación tiene un **MAE de 37,5** pasajeros por hora. Todo se mide en **validación** (octubre a diciembre de 2024, solo filas completas). Test no se toca, se usa en el notebook 07.

Probamos cuatro tipos de modelo:

| Tipo | Modelo | Idea |
|---|---|---|
| `lineal` | Regresión Ridge sobre log(1 + pasajeros), con y sin la combinación hora x tipo de día | Es el modelo más simple que puede aprender la forma del día. Comparando las dos versiones vemos si la combinación hora x tipo de día ayuda |
| `poisson` | Árboles (boosting) con error de Poisson | Los pasajeros se cuentan (0, 1, 2, ...) y el error Poisson está pensado para conteos |
| `log` | Árboles (boosting) sobre log(1 + pasajeros) | La misma idea que el lineal, pero con árboles, que pueden aprender formas más libres |
| `residual` | Árboles (boosting) que aprenden **cuánto corregir** la mediana de 4 semanas | La mediana ya acierta casi siempre. Lo que falta aprender es qué pasa en los días distintos |

También volvemos a calcular tres reglas simples para comparar con las mismas filas: `mediana_4sem`, `perfil_historico` y `regla_por_tipo_dia` (el perfil en feriados y la mediana el resto de los días, la idea que quedó anotada en el 05).

**Cómo elegimos.** Cada modelo tiene "perillas" que hay que decidir antes de entrenar (se llaman **hiperparámetros**, por ejemplo qué tan grande es cada árbol). Probamos unos pocos valores y nos quedamos con el que tiene menos error en validación. Ojo: como elegimos mirando validación, el error en validación del ganador sale un poco más bajo de lo que va a ser en la realidad. El error de verdad lo vemos en test (notebook 07), después de volver a entrenar con train + val.

**Qué se guarda.** Los errores de cada modelo van a la tabla `resultados_modelos` con `etapa = 'modelo'`, al lado de los del 05.

## 0. Preparación

In [0]:
%run ./00_setup

In [0]:
import importlib
import time
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

from bondis import metricas, modelos, variables

for _m in (metricas, modelos, variables):
    importlib.reload(_m)

spark.conf.set("spark.sql.session.timeZone", config.TIMEZONE)
plt.rcParams.update({"figure.figsize": (13, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 200)

OBJ = modelos.OBJETIVO
REFERENCIA = "mediana_4sem"

## 1. Datos

Usamos las mismas filas que en el 05: solo las completas. Train y validación se pasan de Spark a pandas (unas 900 mil y 300 mil filas). Con ese tamaño, scikit-learn (la librería de modelos que usamos en clase) funciona bien y es más fácil de usar que los modelos de Spark.

Las columnas de fin de año (`es_vispera_navidad`, `es_fin_de_anio`, `es_semana_fiestas`) existen solo si se volvió a correr el notebook 03 con el calendario nuevo. Si no están, el notebook sigue sin ellas y avisa. De todas formas, **train no tiene ningún diciembre**: esas columnas valen siempre 0 en train y ningún modelo puede aprender de ellas. Recién se va a ver si sirven en test.

In [0]:
gold = variables.con_tipo_dia(spark.table(config.TABLE_GOLD)).where(F.col("fila_completa"))
leer, faltantes = modelos.columnas_a_leer(gold.columns)
if faltantes:
    print("Columnas opcionales que gold todavía no tiene (se siguen sin ellas):", faltantes)

base = gold.select(*leer, "particion")
tamanos = variables.tamano_linea(gold.where(F.col("particion") == "train")).toPandas()

train_raw = base.where(F.col("particion") == "train").drop("particion").toPandas()
val_raw = (base.where(F.col("particion") == "val").drop("particion").toPandas()
           .merge(tamanos, on="linea", how="left")
           .fillna({"tamano_linea": "chica"}))

categorias = modelos.categorias_desde(train_raw)
train = modelos.preparar(train_raw, categorias)
val = modelos.preparar(val_raw, categorias)
val["tamano_linea"] = val_raw["tamano_linea"].to_numpy()

print(f"train: {len(train):,} filas, {train['fecha'].nunique()} días, {train['linea'].nunique()} líneas")
print(f"val:   {len(val):,} filas, {val['fecha'].nunique()} días ({val['fecha'].min().date()} a {val['fecha'].max().date()})")
sin_linea = (~val["linea"].isin(categorias["linea"])).sum()
print(f"Filas de val con una línea que no está en train: {sin_linea:,}")

## 2. Reglas para comparar

Se calculan de nuevo, ahora en pandas, igual que en el 05. El MAE de `mediana_4sem` tendría que dar 37,5 y el de `perfil_historico`, 37,6. Si no da eso, algo cambió en gold desde que corrimos el 05.

In [0]:
PRED = {}   # para cada modelo, en qué columna de val está su predicción
for nombre, valores in modelos.referencias(train, val).items():
    val[f"pred_{nombre}"] = valores
    PRED[nombre] = f"pred_{nombre}"
    print(f"{nombre:<20} MAE {np.abs(valores - val[OBJ]).mean():6.2f}")

## 3. Modelos lineales

Usamos **Ridge**, que es una regresión lineal común con un pequeño cambio: castiga a los coeficientes muy grandes. Eso hace que el modelo sea más estable cuando hay variables parecidas entre sí.

Predice log(1 + pasajeros) (ver EDA, sección 2) usando:

- El logaritmo de cuatro rezagos: la mediana de 4 semanas, el de 1 día, el de 7 días y el total de la línea el día anterior. Son los cuatro que el EDA dejó como los que no se repiten entre sí.
- Columnas de sí o no del calendario (feriados, Carnaval, Turismo, elecciones, clases) y el clima pronosticado.
- La hora, el tipo de día y el tipo de línea como **categorías**. Se pasan a columnas de 0 y 1 (one hot). Por ejemplo, la hora 8 se convierte en una columna "hora_8" que vale 1 a las 8 y 0 el resto.
- Solo en la versión `interacciones`: una columna por cada par hora y tipo de día (por ejemplo "8_hábil", "8_domingo"). Así cada tipo de día tiene su propia forma a lo largo del día.

`alfa` es qué tan fuerte es el castigo a los coeficientes grandes. Más alto, coeficientes más chicos. Probamos 0,1, 1, 10 y 100 y elegimos con validación. Con 900 mil filas y menos de 200 columnas no esperamos que cambie mucho.

In [0]:
ALFAS = [0.1, 1, 10, 100]

CANDIDATOS = {}   # para cada modelo: (tipo de modelo, sus parámetros)
for interacciones, etiqueta in ((False, "aditivo"), (True, "interacciones")):
    for alfa in ALFAS:
        CANDIDATOS[f"lineal_{etiqueta}_a{alfa}"] = ("lineal", {"interacciones": interacciones, "alfa": alfa})

ajustados = {}


def entrenar(candidatos):
    for nombre, (familia, params) in candidatos.items():
        modelo = modelos.crear_modelo(train.columns, familia, **params)
        pred, seg = modelos.ajustar_y_predecir(modelo, train, val)
        val[f"pred_{nombre}"] = pred
        PRED[nombre] = f"pred_{nombre}"
        ajustados[nombre] = modelo
        print(f"{nombre:<32} {seg:6.1f} s   MAE val {np.abs(pred - val[OBJ]).mean():6.2f}")


entrenar({n: c for n, c in CANDIDATOS.items() if c[0] == "lineal"})

## 4. Árboles con boosting

Un árbol de decisión hace preguntas de sí o no ("¿es día hábil?", "¿la hora es antes de las 9?") hasta llegar a una predicción. **Boosting** quiere decir que se entrenan muchos árboles chicos, uno después del otro, y cada árbol nuevo trata de arreglar los errores que dejaron los anteriores. La predicción final es la suma de todos.

Usamos `HistGradientBoostingRegressor` de scikit-learn. Los árboles usan las variables tal cual (hora, día de la semana, línea, tipo de día, los ocho rezagos, las columnas de sí o no y el clima). `linea`, `tipo_dia` y `tipo_linea` entran como categorías.

Probamos dos tamaños de árbol (`hojas`: 15 o 63 hojas). Lo demás queda fijo: 200 árboles, tasa de aprendizaje 0,1 (qué tanto corrige cada árbol nuevo) y al menos 100 filas por hoja. No usamos la opción de parar antes (`early_stopping`), porque esa opción separa una parte de train **al azar** para controlar, y mezclaría días del futuro con días del pasado.

In [0]:
HOJAS = [15, 63]
for familia in ("poisson", "log", "residual"):
    for hojas in HOJAS:
        CANDIDATOS[f"hgb_{familia}_h{hojas}"] = (familia, {"hojas": hojas, "iteraciones": 200, "tasa": 0.1})

entrenar({n: c for n, c in CANDIDATOS.items() if c[0] != "lineal"})

## 5. Resultados en validación

Se calcula igual que en el 05: sumas por día, métricas a partir de esas sumas e intervalos del 95 % sorteando **días**. Todos los modelos usan los mismos días sorteados. Por eso el intervalo de la **mejora** es el que dice si un modelo le gana de verdad a la mediana de 4 semanas.

In [0]:
est = modelos.estadisticos_por_dia(val, OBJ, PRED, grupos=["tipo_dia", "tamano_linea"])

globales = metricas.agregar_mejora(metricas.metricas(est), REFERENCIA)
intervalos = metricas.bootstrap_dias(est, REFERENCIA, n_rep=1000, seed=config.SEED)
globales = globales.merge(intervalos, on="modelo").sort_values("mae").reset_index(drop=True)


def fmt_intervalo(v, inf, sup, dec=1, pct=False):
    f = (lambda x: f"{100 * x:.{dec}f} %") if pct else (lambda x: f"{x:,.{dec}f}")
    return f"{f(v)} [{f(inf)} a {f(sup)}]"


vista = pd.DataFrame({
    "modelo": globales["modelo"],
    "MAE [IC 95 %]": [fmt_intervalo(*x) for x in globales[["mae", "mae_inf", "mae_sup"]].to_numpy()],
    "RMSE": globales["rmse"].round(1),
    "R²": globales["r2"].round(3),
    "WAPE": (100 * globales["wape"]).round(1).astype(str) + " %",
    "sesgo": globales["sesgo"].round(1),
    "mejora [IC 95 %]": [fmt_intervalo(*x, pct=True) for x in globales[["mejora_mae", "mejora_mae_inf", "mejora_mae_sup"]].to_numpy()],
})
print(vista.to_string(index=False))

**Cómo leer la tabla.**
- Está ordenada de menor a mayor MAE. La **mejora** es 1 menos el MAE del modelo dividido el MAE de `mediana_4sem`. Un 0,10 es un 10 % menos de error.
- Si el intervalo de la mejora incluye el 0, no podemos decir que el modelo le gane a la mediana.
- Si el RMSE es mucho más grande que el MAE, hay pocos errores muy grandes (feriados, elecciones).
- El sesgo dice hacia dónde se equivoca: positivo, predice de más. Negativo, predice de menos. Los modelos con log suelen quedar con sesgo negativo, porque al deshacer el logaritmo la predicción se parece más a una mediana que a un promedio, y en estos datos la mediana es más baja que el promedio.

### 5.1 El mejor de cada tipo de modelo

Miramos cada tipo por separado para ver si la diferencia viene de usar árboles o lineal, o de cómo transformamos lo que queremos predecir.

In [0]:
familia_de = {n: ("lineal_" + n.split("_")[1]) if c[0] == "lineal" else f"hgb_{c[0]}" for n, c in CANDIDATOS.items()}
cand = globales[globales["modelo"].isin(CANDIDATOS)].copy()
cand["familia"] = cand["modelo"].map(familia_de)
mejores = cand.loc[cand.groupby("familia")["mae"].idxmin()].sort_values("mae")
print(mejores[["familia", "modelo", "mae", "rmse", "sesgo", "mejora_mae", "mejora_mae_inf", "mejora_mae_sup"]]
      .round(3).to_string(index=False))

MEJOR = cand.sort_values("mae")["modelo"].iloc[0]
fila = globales.set_index("modelo").loc[MEJOR]
print(f"\nModelo elegido en validación: {MEJOR}")
print(f"  MAE {fila['mae']:.1f}, mejora sobre {REFERENCIA}: {100 * fila['mejora_mae']:.1f} % "
      f"[{100 * fila['mejora_mae_inf']:.1f} % a {100 * fila['mejora_mae_sup']:.1f} %]")
if fila["mejora_mae_inf"] <= 0:
    print("  El intervalo de la mejora incluye el 0. En validación no se puede decir que le gane a la mediana de 4 semanas.")

## 6. ¿Dónde mejora y dónde no?

La mediana de 4 semanas acierta casi siempre, así que un modelo solo puede ganar en los días que no se parecen a las semanas anteriores. Comparamos la mediana, las otras reglas, el mejor de cada tipo y el elegido, separando por tipo de día.

In [0]:
MOSTRAR = [REFERENCIA, "perfil_historico", "regla_por_tipo_dia", *mejores["modelo"]]
MOSTRAR = list(dict.fromkeys(MOSTRAR))
orden = [m for m in globales["modelo"] if m in MOSTRAR]

por_tipo = metricas.agregar_mejora(metricas.metricas(est, por=["tipo_dia"]), REFERENCIA, por=["tipo_dia"])
dias_tipo = est.drop_duplicates(["fecha", "tipo_dia"]).groupby("tipo_dia")["fecha"].nunique().rename("días")
tipos = [t for t in variables.TIPOS_DIA if t in dias_tipo.index]

print("MAE por tipo de día (pasajeros por hora):")
t = por_tipo.pivot(index="modelo", columns="tipo_dia", values="mae").loc[orden, tipos].round(1)
print(pd.concat([dias_tipo[tipos].to_frame().T, t]).to_string(), "\n")

print("Sesgo por tipo de día (> 0: se pasa):")
print(por_tipo.pivot(index="modelo", columns="tipo_dia", values="sesgo").loc[orden, tipos].round(1).to_string())

### 6.1 Por tamaño de línea

El MAE lo deciden las líneas grandes. El WAPE, como es un porcentaje, sirve para comparar grupos de distinto tamaño.

In [0]:
por_tam = metricas.metricas(est, por=["tamano_linea"])
grupos_tam = [g for g in ["grande", "mediana", "chica"] if g in set(por_tam["tamano_linea"])]
for col, titulo in (("mae", "MAE"), ("wape", "WAPE")):
    t = por_tam.pivot(index="modelo", columns="tamano_linea", values=col).loc[orden, grupos_tam]
    print(f"{titulo} por tamaño de línea:")
    print((t * 100).round(1).astype(str).add(" %").to_string() if col == "wape" else t.round(1).to_string(), "\n")

### 6.2 Día por día

El MAE de cada día de validación para la mediana de 4 semanas y para el modelo elegido. Las líneas grises punteadas son días especiales y las rojas son días en que empiezan o terminan las clases.

In [0]:
info_dias = (val[["fecha", "tipo_dia", "nombre_dia_especial", "clases_primaria", "clases_media"]]
             .drop_duplicates("fecha"))
info_dias = info_dias.assign(tipo_dia=info_dias["tipo_dia"].astype(str))

diario = metricas.metricas(est.groupby(["modelo", "fecha"], as_index=False)[metricas.ESTADISTICOS].sum(), por=["fecha"])
fig, ax = plt.subplots()
for m, color in ((REFERENCIA, "tab:gray"), (MEJOR, "tab:blue")):
    s = diario[diario["modelo"] == m].sort_values("fecha")
    ax.plot(s["fecha"], s["mae"], marker=".", ms=4, lw=1, label=m, color=color)
tope = ax.get_ylim()[1]
for _, d in info_dias.dropna(subset=["nombre_dia_especial"]).iterrows():
    ax.axvline(d["fecha"], color="gray", ls=":", lw=1)
    ax.text(d["fecha"], tope, d["nombre_dia_especial"], rotation=90, va="top", ha="right", fontsize=7)

cal = info_dias.sort_values("fecha").set_index("fecha")[["clases_primaria", "clases_media"]]
for nivel in ("primaria", "media"):
    c = cal[f"clases_{nivel}"].astype(int).diff()
    for f, v in c[c != 0].dropna().items():
        etiqueta = f"{'inicio' if v > 0 else 'fin'} de clases ({nivel})"
        ax.axvline(f, color="tab:red", ls="--", lw=1)
        ax.text(f, tope, etiqueta, rotation=90, va="top", ha="right", fontsize=7, color="tab:red")

ax.set(title="MAE por día en validación", ylabel="MAE (pasajeros por hora)")
ax.legend(loc="center left")
plt.show()

peores = (diario[diario["modelo"] == MEJOR].nlargest(10, "mae")
          .merge(info_dias, on="fecha")[["fecha", "tipo_dia", "nombre_dia_especial", "mae", "sesgo"]])
ref_dia = diario[diario["modelo"] == REFERENCIA].set_index("fecha")["mae"].rename("mae_referencia")
peores = peores.merge(ref_dia, left_on="fecha", right_index=True)
print(f"Los 10 días con más error de {MEJOR}:")
print(peores.round(1).to_string(index=False))

## 7. Error en train y en validación: ¿subajuste o sobreajuste?

Hay dos formas de que un modelo salga mal:

- **Subajuste**: el modelo es demasiado simple y no aprende ni siquiera con los datos que vio. Es como estudiar solo el índice del libro. El error es **alto en train y también alto en validación**.
- **Sobreajuste**: el modelo se aprende de memoria los detalles de train, incluso los que fueron casualidad, y después no le va bien con datos nuevos. Es como aprenderse las respuestas del práctico sin entender: en el práctico te va perfecto y en el examen no. El error en train es **mucho más bajo** que en validación.

**Cuidado con una trampa de nuestro problema.** Validación no es una parte al azar de train, es otra época del año (octubre a diciembre). Entonces el error en validación puede ser distinto al de train solo porque son meses distintos, no porque el modelo esté sobreajustado.

Para separar las dos cosas usamos la mediana de 4 semanas como "termómetro". Esa regla no aprende nada, así que no puede sobreajustar. Si su error cambia entre train y validación, es solo por el cambio de época. Entonces calculamos el **exceso**:

$$\text{exceso} = \frac{\text{MAE val del modelo} \,/\, \text{MAE train del modelo}}{\text{MAE val de la mediana} \,/\, \text{MAE train de la mediana}}$$

- Exceso cerca de 1: la diferencia entre train y validación se explica por el cambio de época. No hay sobreajuste.
- Exceso bastante mayor que 1: el modelo empeora en validación más que la mediana. Hay sobreajuste.

### 7.1 Error en train y en validación de cada modelo

El MAE de train se mide sobre una muestra de 200 mil filas, para que sea más rápido.

In [0]:
N_TRAIN_CURVAS = 200_000
sub_train = train.sample(n=min(N_TRAIN_CURVAS, len(train)), random_state=config.SEED)
ref_tr = modelos.mae(sub_train[OBJ], sub_train[modelos.REFERENCIA])
ref_va = modelos.mae(val[OBJ], val[modelos.REFERENCIA])
brecha_ref = ref_va / ref_tr
print(f"{REFERENCIA}: MAE train {ref_tr:.2f}, MAE val {ref_va:.2f}, cociente val/train {brecha_ref:.2f}\n")

diag = modelos.error_train_val(ajustados, list(CANDIDATOS), train, val, n_train=N_TRAIN_CURVAS, semilla=config.SEED)
diag["brecha"] = diag["mae_val"] - diag["mae_train"]
diag["val/train"] = diag["mae_val"] / diag["mae_train"]
diag["exceso"] = diag["val/train"] / brecha_ref
diag["mejora_en_train"] = 1 - diag["mae_train"] / ref_tr
print(diag.sort_values("mae_val").round(3).to_string(index=False))

**Cómo leer la tabla.** `mejora_en_train` dice cuánto le gana el modelo a la mediana en los datos que vio. Si es baja, el modelo no logra aprender ni siquiera train (subajuste, o simplemente no hay mucho más para aprender). Un `exceso` bastante mayor que 1 indica sobreajuste. Si el `exceso` está cerca de 1 y la mejora en train se parece a la mejora en validación, el modelo funciona igual en las dos épocas.

### 7.2 Árboles: el error a medida que se agregan árboles

Cada árbol nuevo corrige a los anteriores. La curva muestra cómo cambia el error al ir sumando árboles. Las líneas punteadas horizontales son el error de la mediana de 4 semanas en train (azul) y en validación (naranja). La línea gris vertical marca el árbol donde el error de validación es el más bajo.

- Si las dos curvas bajan juntas y siguen bajando en el último árbol: **faltan árboles**.
- Si la de train sigue bajando pero la de validación se queda quieta o sube: **sobreajuste**. Conviene usar menos árboles o árboles más chicos.

In [0]:
ARBOLES = [n for n, c in CANDIDATOS.items() if c[0] != "lineal"]
FAMILIAS_ARBOL = ["poisson", "log", "residual"]
curvas = {n: modelos.curva_por_iteraciones(ajustados[n], train, val, n_train=100_000, semilla=config.SEED)
          for n in ARBOLES}

fig, ejes = plt.subplots(len(HOJAS), len(FAMILIAS_ARBOL), figsize=(15, 3.6 * len(HOJAS)), squeeze=False)
for i, hojas in enumerate(HOJAS):
    for j, familia in enumerate(FAMILIAS_ARBOL):
        nombre = f"hgb_{familia}_h{hojas}"
        c, ax = curvas[nombre], ejes[i][j]
        ax.plot(c["iteracion"], c["mae_train"], color="tab:blue", label="train")
        ax.plot(c["iteracion"], c["mae_val"], color="tab:orange", label="validación")
        ax.axhline(ref_tr, color="tab:blue", ls=":", lw=1, alpha=0.7)
        ax.axhline(ref_va, color="tab:orange", ls=":", lw=1, alpha=0.7)
        mejor_it = int(c.loc[c["mae_val"].idxmin(), "iteracion"])
        ax.axvline(mejor_it, color="gray", ls="--", lw=1)
        ax.set(title=f"{familia}, {hojas} hojas (mínimo de val: árbol {mejor_it})", xlabel="árboles", ylabel="MAE",
               ylim=(0, 2.0 * max(ref_tr, ref_va)))
        ax.legend()
plt.tight_layout()
plt.show()

resumen_it = pd.DataFrame([{
    "modelo": n,
    "arboles": int(c["iteracion"].iloc[-1]),
    "mejor_arbol": int(c.loc[c["mae_val"].idxmin(), "iteracion"]),
    "mae_val_minimo": c["mae_val"].min(),
    "mae_val_final": c["mae_val"].iloc[-1],
    "mae_train_final": c["mae_train"].iloc[-1],
} for n, c in curvas.items()])
resumen_it["ganancia_de_cortar"] = resumen_it["mae_val_final"] - resumen_it["mae_val_minimo"]
print(resumen_it.round(3).to_string(index=False))

Si `mejor_arbol` es igual a `arboles`, el error de validación seguía bajando al terminar, y se podría probar con más árboles. Si es mucho menor, los árboles de más solo sobreajustan. `ganancia_de_cortar` es cuánto MAE nos ahorraríamos si paráramos en el mejor árbol. Si es chica, 200 árboles está bien.

### 7.3 Modelos lineales: el error según `alfa`

`alfa` es el castigo a los coeficientes grandes. Con `alfa` muy grande el modelo queda demasiado rígido y no aprende (subajuste). Con `alfa` muy chico es casi una regresión lineal común y podría sobreajustar, aunque con tantas filas y pocas columnas casi no pasa.

In [0]:
lin = diag[diag["modelo"].isin([n for n, c in CANDIDATOS.items() if c[0] == "lineal"])].copy()
lin["alfa"] = lin["modelo"].map(lambda n: CANDIDATOS[n][1]["alfa"])
lin["variante"] = lin["modelo"].map(lambda n: "con interacciones" if CANDIDATOS[n][1]["interacciones"] else "aditivo")

fig, ax = plt.subplots(figsize=(8, 4))
for (variante, g), color in zip(lin.groupby("variante"), ("tab:green", "tab:purple")):
    g = g.sort_values("alfa")
    ax.plot(g["alfa"], g["mae_train"], ls="--", marker="o", color=color, label=f"{variante}, train")
    ax.plot(g["alfa"], g["mae_val"], ls="-", marker="o", color=color, label=f"{variante}, validación")
ax.axhline(ref_va, color="gray", ls=":", lw=1, label=f"{REFERENCIA}, validación")
ax.set(xscale="log", xlabel="alfa", ylabel="MAE", title="Ridge: error según la penalización")
ax.legend(fontsize=8)
plt.show()
print(lin.sort_values(["variante", "alfa"])[["variante", "alfa", "mae_train", "mae_val", "brecha", "exceso"]].round(3).to_string(index=False))

### 7.4 Curva de aprendizaje: ¿más datos ayudarían?

Volvemos a entrenar el modelo elegido con una parte cada vez más grande de los **días** de train (10 %, 25 %, 50 %, 75 % y 100 %). Se sortean días enteros, no filas sueltas. Cada vez medimos el error en esos mismos días y en validación.

- Si las dos curvas se juntan y quedan **planas y altas**: más datos no ayudan. El problema es el modelo (subajuste) o que hay cosas que no se pueden predecir.
- Si hay una **distancia entre las curvas que se achica** al agregar datos: más datos ayudarían, y el modelo hoy está un poco sobreajustado.
- Cuidado: con pocos días, train no tiene todos los tipos de día. Por eso, del lado izquierdo del gráfico la distancia mezcla sobreajuste con falta de ejemplos.

In [0]:
familia_m, params_m = CANDIDATOS[MEJOR]
curva_ap = modelos.curva_aprendizaje(
    lambda: modelos.crear_modelo(train.columns, familia_m, **params_m),
    train, val, fracciones=(0.1, 0.25, 0.5, 0.75, 1.0), semilla=config.SEED)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curva_ap["dias"], curva_ap["mae_train"], marker="o", color="tab:blue", label="train")
ax.plot(curva_ap["dias"], curva_ap["mae_val"], marker="o", color="tab:orange", label="validación")
ax.axhline(ref_tr, color="tab:blue", ls=":", lw=1, alpha=0.7, label=f"{REFERENCIA}, train")
ax.axhline(ref_va, color="tab:orange", ls=":", lw=1, alpha=0.7, label=f"{REFERENCIA}, validación")
ax.set(xlabel="días de train usados", ylabel="MAE", title=f"Curva de aprendizaje: {MEJOR}")
ax.legend(fontsize=8)
plt.show()
print(curva_ap.round(3).to_string(index=False))

**Qué muestra la sección 7.** Completar en las conclusiones: si algún modelo sobreajusta (exceso bastante mayor que 1, o error de validación que sube al agregar árboles), si hay subajuste (error alto en train y poca mejora sobre la mediana) y si más datos ayudarían. Train tiene solo de enero a setiembre de 2024, así que el modelo nunca vio un octubre, noviembre o diciembre. Eso limita lo que puede aprender, y se arregla en parte al volver a entrenar con train + val.

## 8. ¿Qué variables usa más el modelo?

Usamos la **importancia por permutación**. La idea es simple: tomamos una muestra de validación y **mezclamos al azar** los valores de un grupo de variables, como si barajáramos esa columna. Así esa variable deja de tener sentido. Después medimos cuánto sube el MAE:
- Si sube mucho, el modelo dependía mucho de esa variable.
- Si casi no sube, el modelo no la usaba mucho.

Las variables se mezclan en grupos porque varias dicen casi lo mismo (los rezagos, el promedio y la mediana de 4 semanas). Si mezcláramos una sola, el modelo usaría las otras y parecería que no importa.

Es una medida aproximada. No dice que una variable **cause** más pasajeros. Y si un grupo da importancia baja, puede ser que no sirva o que otro grupo diga lo mismo.

In [0]:
muestra = val.sample(n=min(100_000, len(val)), random_state=config.SEED)
imp = modelos.importancia_por_grupos(ajustados[MEJOR], muestra, modelos.grupos_de_variables(val.columns),
                                     n_rep=3, seed=config.SEED)
print(f"MAE de {MEJOR} en la muestra sin desordenar: {imp.attrs['mae_base']:.2f}\n")
print(imp[["grupo", "mae_permutado", "aumento_mae", "aumento_pct"]].round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.barh(imp["grupo"][::-1], imp["aumento_mae"][::-1], color="tab:blue")
ax.set(title=f"Aumento del MAE al desordenar cada grupo ({MEJOR})", xlabel="aumento del MAE (pasajeros por hora)")
plt.show()

## 9. Guardar los resultados

Se guardan en `resultados_modelos` con `etapa = 'modelo'`, al lado de las reglas del 05. Si se vuelve a correr este notebook, se reemplazan solo las filas de `modelo`. Se agregan dos columnas: `parametros` (con qué valores se entrenó cada modelo) y `seleccionado` (cuál fue el elegido).

In [0]:
ETAPA = "modelo"
salida = globales[globales["modelo"].isin(CANDIDATOS)].copy()
salida = salida.assign(
    etapa=ETAPA, conjunto="val", referencia=REFERENCIA,
    parametros=[f"{CANDIDATOS[m][0]} {CANDIDATOS[m][1]}" for m in salida["modelo"]],
    seleccionado=(salida["modelo"] == MEJOR).to_numpy(),
    fecha_ejecucion=datetime.now().strftime("%Y-%m-%d %H:%M"),
)
columnas = ["etapa", "conjunto", "modelo", "referencia", "filas", "mae", "mae_inf", "mae_sup", "rmse",
            "rmse_inf", "rmse_sup", "r2", "wape", "sesgo", "mejora_mae", "mejora_mae_inf", "mejora_mae_sup",
            "parametros", "seleccionado", "fecha_ejecucion"]
no_numericas = ("etapa", "conjunto", "modelo", "referencia", "fecha_ejecucion", "filas", "parametros", "seleccionado")
salida = salida[columnas].astype({c: "float64" for c in columnas if c not in no_numericas})

escritor = spark.createDataFrame(salida).write.format("delta").mode("overwrite").option("mergeSchema", "true")
if spark.catalog.tableExists(config.TABLE_RESULTADOS):
    escritor.option("replaceWhere", f"etapa = '{ETAPA}'").saveAsTable(config.TABLE_RESULTADOS)
else:
    escritor.saveAsTable(config.TABLE_RESULTADOS)

print(spark.table(config.TABLE_RESULTADOS).where(F.col("etapa") == ETAPA)
      .select("modelo", "mae", "rmse", "r2", "mejora_mae", "seleccionado").toPandas().round(3).to_string(index=False))

## Conclusiones

Completar con los resultados, como en el 05. Resultados en validación (01/10/2024 a 31/12/2024, solo filas completas).

| Pregunta | Resultado |
|---|---|
| ¿Qué modelo se equivoca menos? ¿De qué tipo es? | |
| ¿Cuánto mejora sobre `mediana_4sem` (MAE 37,5)? ¿El intervalo incluye el 0? | |
| ¿La combinación hora x tipo de día mejora al modelo lineal? ¿Cuánto? | |
| ¿Qué versión de árboles funciona mejor (Poisson, log o residual)? | |
| ¿El modelo elegido le gana a `regla_por_tipo_dia`? Si no, ¿para qué sirve entrenar? | |
| ¿En qué tipo de día mejora más? ¿En cuál empeora? | |
| ¿Mejora igual en las líneas grandes, medianas y chicas? | |
| ¿Qué grupo de variables pesa más? ¿Hay alguno que no sirve? | |
| ¿Hay subajuste o sobreajuste? ¿Cuántos árboles conviene usar? ¿Más datos ayudarían? | |
| ¿Qué errores siguen sin resolver (días especiales, elecciones, diciembre)? | |

**Límites.** Elegimos el modelo mirando validación, así que su error ahí sale un poco mejor de lo real. Train no tiene ningún diciembre, así que las variables de fin de año no se pueden aprender todavía, y en train hay muy pocos feriados y elecciones. En el notebook 07 se vuelve a entrenar con train + val (todo 2024) y se prueba en test.